# 스토리형 콘텐츠 산업 뉴스레터 에이전트 (Colab)

N036 강의(뉴스레터 에이전트 만들기)의 5단계 파이프라인 "수집→선별→요약→검수→발행"을
그대로 적용해, 아래 세 트랙의 소식을 모아 Discord로 발행합니다.

- **콘텐츠산업뉴스**: 웹소설·웹툰·숏폼·드라마 산업 동향
- **채용**: 위 분야와 연계된 채용 공고
- **지원사업**: 관련 정부·공공기관 지원사업 공고

## 실행 순서
위에서부터 순서대로 실행하세요. **처음 실행할 때는 반드시 `DRY_RUN = True`로 먼저 돌려서
결과를 눈으로 확인한 뒤에만 `False`로 바꿔 실제 Discord에 발행하세요.**


## 0. 환경 설정

### 0-1. 라이브러리 설치


In [ ]:
!pip install -q openai langgraph pydantic trafilatura requests feedparser googlenewsdecoder

### 0-2. API 키·웹훅을 Colab 보안 비밀로 등록

**절대 키나 웹훅 URL을 코드 셀에 직접 문자열로 적지 마세요.** 채팅에 붙여넣는 것도 마찬가지입니다 —
웹훅 URL은 인증 절차가 없어서, 주소를 아는 사람은 누구나 그 채널에 글을 쓸 수 있습니다
(N036 챕터10 "웹훅 URL = 열쇠" 참고).

1. Colab 왼쪽 사이드바 **🔑(보안 비밀)** 클릭
2. 두 개를 등록합니다:
   - 이름 `OPENAI_API_KEY` — 값: OpenAI API 키
   - 이름 `DISCORD_WEBHOOK_URL` — 값: 발행할 Discord 채널의 웹훅 URL
3. 둘 다 **노트북 액세스** 토글을 켭니다
4. 아래 셀을 실행합니다


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["DISCORD_WEBHOOK_URL"] = userdata.get("DISCORD_WEBHOOK_URL")
    print("Colab 보안 비밀에서 키/웹훅을 불러왔습니다.")
except Exception as e:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        print("확인:", "OPENAI_API_KEY" in os.environ, "DISCORD_WEBHOOK_URL" in os.environ)
    except ImportError:
        print("키를 찾지 못했습니다:", e)

print("OPENAI_API_KEY 설정 여부:", "설정됨" if os.environ.get("OPENAI_API_KEY") else "비어 있음")
print("DISCORD_WEBHOOK_URL 설정 여부:", "설정됨" if os.environ.get("DISCORD_WEBHOOK_URL") else "비어 있음")

### 0-3. 안전장치 — 실제 발행 여부

`DRY_RUN = True`이면 무엇이 나갈지만 콘솔에 보여주고 Discord로는 아무것도 보내지 않습니다.
처음엔 반드시 True로 실행해서 결과를 확인하세요.


In [ ]:
DRY_RUN = True  # 실제 발행하려면 맨 마지막 발행 셀에서 False로 바꿔서 그 셀만 다시 실행하세요

## 1. 트랙 설정 — 무엇을, 왜 다르게 볼 것인가 (N036 챕터13 audience.yaml 원칙)

트랙마다 검색어와 "무엇을 관문/등급 기준으로 볼지"를 다르게 둡니다. 코드(아래 파이프라인)는
전혀 안 건드리고, 이 설정만 고치면 트랙을 추가·수정할 수 있습니다.


In [ ]:
TRACKS = {
    "콘텐츠산업뉴스": {
        "queries": ["웹툰 산업", "웹소설 시장", "숏폼 드라마", "OTT 드라마 제작"],
        "score_criteria": (
            "구체적인 수치·계약·편성·투자·정책 변화가 있는 소식을 높게, "
            "단순 홍보·행사 안내는 낮게 평가하세요."
        ),
        "has_deadline": False,
        "discord_color": 0x4A90D9,
    },
    "채용": {
        "queries": ["웹툰 웹소설 채용", "콘텐츠 기획 채용", "숏폼 PD 채용", "드라마 제작 채용"],
        "score_criteria": (
            "웹소설·웹툰·숏폼·드라마 제작/기획/전략 직무와 관련도가 높은 공고를 높게, "
            "무관한 직무나 사원·인턴급 단순 채용은 낮게 평가하세요."
        ),
        "has_deadline": True,
        "discord_color": 0xE0A35F,
    },
    "지원사업": {
        "queries": ["웹툰 웹소설 지원사업", "콘텐츠 창작 지원사업", "문화체육관광부 콘텐츠 공모"],
        "score_criteria": (
            "웹소설·웹툰·숏폼·드라마 창작/제작사가 실제로 신청할 수 있는 지원사업을 높게, "
            "관련 없는 분야나 이미 마감된 사업은 낮게 평가하세요."
        ),
        "has_deadline": True,
        "discord_color": 0x6FBF73,
    },
}

G1_MIN_CHARS = 600  # N036 챕터4의 G1 관문 기준
LOOKBACK_ITEMS_PER_QUERY = 6

## 2. 수집(Collect) — Google 뉴스 검색(RSS)

N036 챕터3의 사다리 ②(RSS/Atom)를 그대로 씁니다. Google 뉴스 검색 RSS는 인증 없이 쓸 수
있고, 검색어 기반이라 특정 사이트 구조 변경에 흔들리지 않습니다.


In [ ]:
import feedparser
from urllib.parse import quote

def search_news(query, max_items=LOOKBACK_ITEMS_PER_QUERY):
    url = f"https://news.google.com/rss/search?q={quote(query)}&hl=ko&gl=KR&ceid=KR:ko"
    feed = feedparser.parse(url)
    items = []
    for e in feed.entries[:max_items]:
        items.append({
            "title": e.title,
            "link": e.link,
            "published": e.get("published", ""),
        })
    return items

def collect_track(track_name):
    seen_links = set()
    candidates = []
    for q in TRACKS[track_name]["queries"]:
        for item in search_news(q):
            key = item["link"].split("?")[0]  # URL 정규화로 중복 제거 (N036 챕터5)
            if key in seen_links:
                continue
            seen_links.add(key)
            item["track"] = track_name
            candidates.append(item)
    return candidates

all_candidates = {}
for track in TRACKS:
    found = collect_track(track)
    all_candidates[track] = found
    print(f"[{track}] {len(found)}건 수집")

## 3. 실제 기사 URL 해석 + 본문 확보 + G1 관문

**여기서 실제로 부딪힌 문제 하나**: Google 뉴스 RSS의 `link`는 진짜 기사 주소가 아니라
`news.google.com/rss/articles/...` 형태의 리다이렉트 래퍼입니다. `trafilatura`로 이
래퍼 주소를 그대로 열어보면 자바스크립트로만 진짜 기사로 넘어가는 빈 껍데기 페이지라
본문 추출이 **항상 실패**합니다(직접 테스트로 확인한 실제 문제입니다). 그래서
`googlenewsdecoder`로 먼저 진짜 기사 URL을 풀어낸 뒤에만 본문을 추출합니다.

그다음 본문이 요약 재료로 쓸 수 있을 만큼(600자 이상) 확보됐는지 확인합니다 — 재료 없이
요약을 시키면 모델이 제목만 보고 지어냅니다(N036 챕터4·8의 G1 관문).

⏱️ URL 하나 해석에 약 1~2초가 걸려서, 후보가 수십 건이면 이 셀만 1~3분 정도 걸릴 수
있습니다(API 비용은 없습니다 — 아직 OpenAI를 호출하지 않는 단계입니다).


In [ ]:
import trafilatura
from googlenewsdecoder import gnewsdecoder

def resolve_and_extract(wrapped_link):
    decoded = gnewsdecoder(wrapped_link, interval=1)
    if not decoded.get("status"):
        return None, None
    real_link = decoded["decoded_url"]
    downloaded = trafilatura.fetch_url(real_link)
    if not downloaded:
        return real_link, None
    text = trafilatura.extract(downloaded)
    if not text or len(text) < G1_MIN_CHARS:
        return real_link, None
    return real_link, text

for track, items in all_candidates.items():
    seen_real_links = set()
    passed = []
    for item in items:
        real_link, body = resolve_and_extract(item["link"])
        if not body:
            continue
        if real_link in seen_real_links:  # 서로 다른 검색어가 같은 기사를 중복 반환하는 경우 제거
            continue
        seen_real_links.add(real_link)
        item["link"] = real_link  # 이후 단계·최종 발행에는 항상 실제 기사 URL을 씀
        item["body"] = body
        passed.append(item)
    print(f"[{track}] G1 관문 통과: {len(passed)} / {len(items)}건")
    all_candidates[track] = passed

## 4. 선별(Score) — 트랙별 전체 비교 상대평가

후보 수가 많지 않으므로(N036 6장 "③ 전체 비교") 예선/본선 없이, 트랙 안에서 한 번에
비교해 관련도 점수를 매기고 상위 N건만 남깁니다.


In [ ]:
from pydantic import BaseModel, Field
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"
TOP_N_PER_TRACK = 4

class ScoredCandidate(BaseModel):
    index: int = Field(description="후보 목록에서의 0부터 시작하는 순번")
    score: int = Field(description="0~100 관련도 점수")
    reason: str = Field(description="이 점수를 준 한 줄 이유(한국어)")

class ScoreResult(BaseModel):
    scored: list[ScoredCandidate]

def score_track(track_name, items):
    if not items:
        return []
    listing = "\n".join(f"{i}: {it['title']}" for i, it in enumerate(items))
    prompt = (
        f"다음은 '{track_name}' 트랙 후보 목록입니다. 각 후보에 0~100 관련도 점수를 매기세요.\n"
        f"평가 기준: {TRACKS[track_name]['score_criteria']}\n"
        "같은 사건을 다룬 후보는 하나만 높은 점수를 주고 나머지는 낮게 매기세요.\n\n"
        f"{listing}"
    )
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=ScoreResult,
    )
    scored = completion.choices[0].message.parsed.scored
    ranked = sorted(scored, key=lambda s: -s.score)
    top_indices = [s.index for s in ranked[:TOP_N_PER_TRACK]]
    return [items[i] for i in top_indices if i < len(items)]

selected = {}
for track, items in all_candidates.items():
    top = score_track(track, items)
    selected[track] = top
    print(f"[{track}] 최종 선정: {len(top)}건")
    for it in top:
        print("  -", it["title"])

## 5. 요약 스키마 정의 (검증 가능성 기준으로 칸 분리 — N036 챕터8)


In [ ]:
import re
from typing import Optional

_KOREAN_RE = re.compile(r"[가-힣]")

def has_korean(text):
    return bool(_KOREAN_RE.search(text or ""))

class ContentCard(BaseModel):
    headline: str = Field(description="카드용 한국어 헤드라인. ~합니다체로 간결하게.")
    summary: str = Field(description="세 문장짜리 한국어 요약. 원문에 있는 사실만 담습니다.")
    why: str = Field(description="왜 중요한지 한국어 한 문장. summary에 있는 내용만 근거로 삼고 새 사실을 추가하지 않습니다.")
    deadline: Optional[str] = Field(default=None, description="채용·지원사업 트랙만: 마감일(YYYY-MM-DD). 본문에 명시가 없으면 null.")

SYSTEM_PROMPT = "당신은 웹소설·웹툰·숏폼·드라마 업계 종사자를 위한 뉴스레터 기자입니다. 주어진 본문만 근거로 삼아 답합니다."

def draft_card(item, extra_instruction=None):
    track = item["track"]
    ask_deadline = "본문에 마감일이 있으면 deadline 필드에 YYYY-MM-DD로 적으세요. " if TRACKS[track]["has_deadline"] else ""
    user_prompt = (
        f"다음 글을 바탕으로 headline, summary, why를 작성하세요. {ask_deadline}\n\n"
        f"제목: {item['title']}\n본문:\n{item['body'][:6000]}"
    )
    if extra_instruction:
        user_prompt += f"\n\n{extra_instruction}"
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=ContentCard,
    )
    return completion.choices[0].message.parsed

# N036 챕터8 실습(2026-09-14)에서 실제로 확인한 문제 반영: headline+summary를 합쳐서
# 검사하면 한쪽 필드의 실패를 다른 쪽이 가릴 수 있다 — 반드시 필드별로 따로 검사한다.
def draft_card_with_korean_retry(item, max_retries=1):
    retries = 0
    card = draft_card(item)
    while (not (has_korean(card.headline) and has_korean(card.summary))) and retries < max_retries:
        retries += 1
        card = draft_card(
            item,
            extra_instruction="headline과 summary가 한국어가 아니었습니다. 두 필드를 반드시 한국어로 다시 쓰세요.",
        )
    return card, retries

## 6. LangGraph `Send`로 팬아웃 (N036 챕터8)

선정된 후보 수만큼 요약 노드를 동시에 띄웁니다. 워커는 자기가 맡은 후보만 알고, 다른
워커의 존재를 모릅니다.


In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class GraphState(TypedDict):
    selected: list
    drafted: Annotated[list, operator.add]
    log: Annotated[list, operator.add]

def route_to_draft(state: GraphState):
    return [Send("draft", {"item": it}) for it in state["selected"]]

def draft_node(payload):
    item = payload["item"]
    try:
        card, retries = draft_card_with_korean_retry(item)
    except Exception as e:
        return {"drafted": [], "log": [f"실패: {item['title'][:30]}... ({e})"]}
    record = card.model_dump()
    record["track"] = item["track"]
    record["link"] = item["link"]
    msg = f"완료: {item['title'][:30]}..." + (f" (한국어 재시도 {retries}회)" if retries else "")
    return {"drafted": [record], "log": [msg]}

builder = StateGraph(GraphState)
builder.add_node("draft", draft_node)
builder.add_conditional_edges(START, route_to_draft, ["draft"])
builder.add_edge("draft", END)
graph = builder.compile()

flat_selected = [it for items in selected.values() for it in items]
result = graph.invoke({"selected": flat_selected, "drafted": [], "log": []})

print(f"선택 {len(flat_selected)}건 → 취재 완료 {len(result['drafted'])}건\n")
for line in result["log"]:
    print("-", line)

drafted_cards = result["drafted"]

## 7. 검수 — 마감일 형식·경과 여부 확인 (채용·지원사업 트랙만)


In [ ]:
from datetime import date

def check_deadline(card):
    if not TRACKS[card["track"]]["has_deadline"]:
        return "해당없음"
    d = card.get("deadline")
    if not d:
        return "마감일 확인 필요(본문에 명시 없음)"
    try:
        y, m, day = map(int, d.split("-"))
        parsed = date(y, m, day)
    except Exception:
        return f"마감일 형식 이상: {d!r}"
    if parsed < date.today():
        return f"마감 지남({d}) — 발행에서 제외 권장"
    return f"정상({d})"

for card in drafted_cards:
    status = check_deadline(card)
    card["deadline_status"] = status
    if card["track"] != "콘텐츠산업뉴스":
        print(f"[{card['track']}] {card['headline'][:30]}... → {status}")

# 마감 지난 채용/지원사업 카드는 발행 목록에서 제외
final_cards = [
    c for c in drafted_cards
    if not (TRACKS[c["track"]]["has_deadline"] and "마감 지남" in c["deadline_status"])
]
print(f"\n최종 발행 대상: {len(final_cards)} / {len(drafted_cards)}건")

## 8. 발행(Publish) — Discord 웹훅

`DRY_RUN`이 True면 무엇이 나갈지만 보여주고 실제로는 보내지 않습니다. 트랙별로 다른 색상의
embed로 구성하고, Discord 한도(embed 10개/제목 256자/본문 4096자)를 지킵니다.


In [ ]:
import requests
from datetime import datetime

def build_embed(card):
    desc = card["summary"]
    if TRACKS[card["track"]]["has_deadline"] and card.get("deadline"):
        desc += f"\n\n🗓️ 마감: {card['deadline']}"
    desc += f"\n\n💡 {card['why']}"
    return {
        "title": card["headline"][:256],
        "description": desc[:4096],
        "url": card["link"],
        "color": TRACKS[card["track"]]["discord_color"],
        "footer": {"text": card["track"]},
    }

def publish(cards, dry_run=True):
    if not cards:
        payload_text = "오늘은 조건에 맞는 신규 소식이 없습니다."
        if dry_run:
            print(f"[dry-run] {payload_text}")
            return
        requests.post(os.environ["DISCORD_WEBHOOK_URL"], json={"content": payload_text})
        return

    for i in range(0, len(cards), 10):  # Discord embed 한도: 메시지당 10개
        chunk = cards[i:i + 10]
        embeds = [build_embed(c) for c in chunk]
        payload = {
            "content": f"📰 스토리형 콘텐츠 산업 뉴스레터 — {datetime.now().strftime('%Y-%m-%d')}"
            if i == 0 else None,
            "embeds": embeds,
        }
        if dry_run:
            print(f"[dry-run] embed {len(embeds)}개 — 보내지 않습니다")
            for e in embeds:
                print(f"  - [{e['footer']['text']}] {e['title']}")
        else:
            resp = requests.post(os.environ["DISCORD_WEBHOOK_URL"], json=payload)
            print("상태 코드:", resp.status_code, "(204면 성공)")

publish(final_cards, dry_run=DRY_RUN)

### 실제로 Discord에 발행하려면

위 결과를 확인하고 문제없으면, 맨 위 "0-3. 안전장치" 셀로 돌아가 `DRY_RUN = False`로 바꿔
**그 셀만** 다시 실행한 뒤, 바로 위 발행 셀을 다시 실행하세요.
